<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/day29_grpo_intuition_tw_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 29 筆記：GRPO 直覺（對比 PPO / DPO）

> 筆記重點「為什麼要這樣設計，以及這個設計的TRADEOFF在哪」。

---

## 1. Reward 從哪裡來——兩種路線

想像你在教一個學生寫作文，你有兩種完全不同的評分方式：

**方式一：對答案（Rule-based Reward）**

出一題數學題「3 × 24 = ?」，答案就是 72。這種題目**不需要人去感覺好不好**，寫一支程式就能判斷：輸出如果是 72，給 1 分；不是 72，給 0 分。沒有模糊地帶，不需要任何「模型」去打分，一行 `if answer == 72: reward = 1 else: reward = 0` 就搞定。

**方式二：找裁判（Reward Model / Preference）**

出一題「寫一篇關於友情的短文」，這種題目**沒有標準答案**，沒辦法用程式判斷好壞。what now？我們可以事先訓練一個「裁判模型」by 餵它大量「人類覺得這篇比那篇好」的資料，讓它學會模仿人類的品味。訓練完之後，這個裁判就**凍結**（不再更新），之後每次要打分，就把學生的作文丟給裁判，裁判讀完吐出一個分數（例如 0.8 分）。

兩者整理成表：

| | Rule-based Reward | Reward Model（Preference） |
|---|---|---|
| 誰打分 | 寫死的規則（答案對不對、code 跑不跑得過） | 一個訓練好的、**凍結**的裁判模型 |
| 分數性質 | 客觀、可驗證，人人算出來都一樣 | 主觀、模仿人類偏好，換個裁判可能打分不同 |
| 代表案例 | DeepSeekMath / DeepSeek-R1（GRPO） | InstructGPT（PPO） |
| 額外成本 | 幾乎零，一行判斷式 | 要先花錢、花時間訓練一個 reward model |

**DeepSeekMath 走的是第一條路**：一道數學題，模型吐出一長串「思考過程 + 最終答案」，系統只看**最後那個數字**是不是 72。對 → 這整條輸出得 1 分；錯 → 得 0 分。

讀到這裡我卡到了一個地方：**中間過程沒有標準答案可以比對。** 模型可能用了 5 種不同的算法路徑得到 72，沒有人告訴它「你這一步該怎麼寫」。那這 1 分要怎麼分給中間那些 token（思考過程裡的每一個字）？

解決的方式是：**broadcast（廣播）**——既然這整條輸出是一個整體，那我就假設「這條路徑上的每一個 token 都有功勞（或都有責任）」，把**整條輸出的 advantage 原封不動複製貼上**給輸出裡的每一個 token。不是分 attention、不是按重要性加權，就是單純每個位置都填上同一個數字。

> ⚠️ 順序不要搞反：被廣播的是**第 2 節算完的 Â_i**（+0.577 / −1.732 那個），不是原始的 r_i（1 或 0）。
> 流程是：**先**對整組算 mean / std → 得到每條輸出的 Â_i → **再**把 Â_i 廣播給該條輸出的每個 token。

這也是為什麼 GRPO 不需要「逐 token 監督」——我把整條的評價粗暴地複製給全部位置就夠用了。

---

## 2. What is Group Relative？

TL;DR：我不需要老師在我旁邊預測我的成績，我直接跟我的 cohort 比。如果我的分數大於平均，那我一定是做對了什麼，模型會獎勵我繼續做。

以下是個簡單的例子。

假設題目是「3 × 24 = ?」，GRPO 設定 G = 4（一次採樣 4 份答案，實際訓練通常是 8~64，這裡用 4 方便手算）。模型針對同一題，生成 4 種不同的解法：

| 第幾份 | 模型寫的東西 | 最終答案 | Reward r_i |
|---|---|---|---|
| o_1 | 3×24 = 3×20+3×4 = 60+12 = 72 | 72 | 1 |
| o_2 | 3×24 = 3×25−3 = 75−3 = 72 | 72 | 1 |
| o_3 | 3×24 = 3×20+3×4 = 60+8 = 68 | 68 | 0 |
| o_4 | 3×24 = 72（沒寫過程直接猜） | 72 | 1 |

**第一步：算這組的平均分和標準差**

- mean = (1+1+0+1) / 4 = 0.75
- std ≈ 0.433（這組數字的離散程度）

**第二步：算每一份的「優勢」（Â_i）**

Â_i = (r_i − mean) / std

- o_1：(1 − 0.75) / 0.433 ≈ **+0.577** → 正的，加強這條路徑（哎呀，我一定是做對了什麼，繼續！）
- o_2：同上，**+0.577** → 加強（我也是耶，我繼續！）
- o_3：(0 − 0.75) / 0.433 ≈ **−1.732** → 負的，削弱這條路徑（欸？！你先等等）
- o_4：**+0.577** → 加強（雖然過程亂寫，但答案剛好對，還是被加強。我賽到答案，但反正對了就是對的，我繼續！）

> o_4 這一格很重要，先記著。它就是 rule-based reward 的漏洞在小規模下的長相——**答案對，但過程是空的，照樣滿分**。第 8 節會回來找它算帳。

**重點觀察**：這個 0.75 的平均值（baseline），**不是任何人事先告訴模型的，也不是另外一個模型猜出來的**——它是這一批 4 個樣本自己「考」出來的真實平均分。PPO 需要一個額外的 value model 去**猜**「這題大概能拿幾分」當作 baseline；GRPO 直接讓模型自己考 G 次，用這 G 次的**真實成績**當 baseline，零額外模型、零猜測。

這就是「Group（一組樣本）」+「Relative（相對於這組的平均）」兩個詞合起來的意思。

---

## 3. 我搞錯的地方 ①：PPO 的 Critic 為什麼「貴」

我一開始的直覺是：「因為要對每個 token 都打分，token 那麼多，跑起來一定很貴很慢。」**這個方向是錯的**，值得花時間搞懂為什麼錯。

**Transformer本身就不需要「一個字一個字慢慢想」**

RNN（Transformer 之前的架構）確實是逐字處理：算完第 1 個字才能算第 2 個字，像排隊一樣。但 Transformer 的 attention 機制是**一次把整條句子平行算完**——這正是它比 RNN 快、能取代 RNN 的原因。

所以「輸出的每個位置都要有一個結果」這件事，在 Transformer 裡**幾乎是免費的副產品**：模型做一次 forward pass，每個位置的 hidden state（內部表示向量）就已經算好放在那裡了。Value model 要做的事，只是在每個位置的 hidden state 後面，多接一個很小的線性層（把幾百維的向量壓成一個數字），這個計算量小到可以忽略。

**真正貴的地方，是這兩件事：**

**(1) Value model 是一整個獨立的、跟 policy model 差不多大的網路。**
不是「多接一層」那麼簡單，是整組獨立的 transformer 層（自己的 attention、自己的 FFN），要做**自己的一整次 forward + backward**。訓練時，參數、梯度、optimizer states（Adam 的話還要存動量）這三樣東西，**全部再多一份**。Day 27 QLoRA 那份記憶體對照表裡看到的「一個模型佔多少記憶體」，這裡直接乘以 2 份模型。

**(2) 監督訊號極度稀疏。**
Reward model 通常只在句尾給一個分數，其他位置全是 0。但 value function 的任務是：在**每一個** token 位置，都要準確估計「從這裡接下去，平均能拿多少分」。問題來了——你只有句尾一個監督訊號（1 個數字），卻要教會模型在 100 個位置都給出準確估計，訊號密度差了 100 倍。

這是「難學」的問題，跟「貴不貴」是兩件不同的事，但論文裡兩個問題是一起被提出來的。

GRPO 一次把這兩個問題都繞過去了：**不訓練 value model**，改用同一組採樣自己的統計量（第 2 節算的那個 mean/std）當 baseline。省掉了整個獨立網路，也不用煩惱「稀疏訊號怎麼學」——因為根本不用學，是直接算出來的。

▲ 參考 `day27_qlora_memory` 詳解關於 memory cost。

---

## 4. 我搞錯的地方 ②：不是「跟正確答案比」就叫監督式學習(RL vs SL)

我這裡又有一個誤解：「GRPO 最後不也是看答案嗎?那不就是監督式學習（SL）嗎？」——yes and NO，因為關鍵不在「有沒有正確答案」，而在**「誰寫出了通往答案的那條路」**。

**SFT（Supervised Fine-Tuning，屬於 SL）的做法：**

老師不只給題目，**連解題過程也一起給你**。例如訓練資料長這樣：

> 問題：3×24 是多少？
> 標準答案：3×24 = 3×20 + 3×4 = 60 + 12 = **72**

模型的任務是：**模仿這串文字**，學會下次遇到類似問題，也照這個方式一步一步寫。它不需要「想出」這個解法，只需要「記住並複製」這個解法的**風格**。訓練時用的 loss 也很直接：比對模型自己吐出來的每個字，跟標準答案的每個字，是不是一樣的機率分布（cross-entropy）。**學的是「樣子」**——像不像老師教的那樣寫。嗯……大概就是所謂的填鴨式教育啦。

**GRPO（RL）的做法：**

老師**只給題目和終點**（告訴你答案是 72），完全不給過程。

> 問題：3×24 是多少？
> （沒有提供任何解法示範）

模型必須自己生成一整套解法，可能對、可能錯。錯了（像上面例子的 o_3，算成 68）就得 0 分，沒有人告訴它錯在哪一步、該怎麼改；對了就得 1 分。模型要靠**大量嘗試**（這一題可能被練習幾千次、幾萬次），自己去試出「原來這樣拆解比較不會算錯」這種內化的策略。**學的是「邏輯」**——不是記住某個範例，而是發展出一套自己能重複使用的解題方法。

**這就是為什麼答案（72）不算「監督式學習裡的標籤」**：72 只是終點站的座標，通往那裡的路完全是模型自己走出來的，沒有人手把手教。這條「自己試錯、自己摸出路徑」的過程，正是 RL 能訓練出真正推理能力，而不只是表面模仿的原因。

---

##   5.★  DPO 放在哪裡

DPO（Direct Preference Optimization）是介於 SFT 跟 GRPO 之間的另一種做法，先講它怎麼運作：

給模型一對答案，一好一壞：

> A（好答案）：3×24 = 3×20+3×4 = 60+12 = 72
> B（壞答案）：3×24 = 3×20+3×5 = 60+15 = 75（算錯）

DPO 用一條數學公式，直接調整模型參數，讓它「以後更常說出像 A 的話，更少說出像 B 的話」。不需要裁判模型即時打分，也不需要 GRPO 那種一次採樣 G 份的迴圈，訓練起來簡單、便宜、穩定很多。

**我的第一個想法是「DPO 沒有推理能力」，但其實有沒有推理能力取決於 A 裡面有沒有推理過程。**

- 如果訓練資料裡的 A（好答案）本身就寫了完整的解題步驟，DPO 確實能學到這個推理風格。
- 但 DPO 有一個**根本性的弱點：它不會自我探索**。它永遠只能在人類已經準備好的 A/B 這兩個選項之間二選一，**沒辦法自己生出一個 A 和 B 都沒想到的第三種解法**。

**RL（GRPO）強在哪裡：**

回頭看第 2 節的例子——G=4 的時候，模型自己嘗試了 4 種不同寫法（o_1 到 o_4），其中可能有一種是**老師從來沒示範過的巧妙解法**（Eureka!）。只要這條路徑最後算對了，GRPO 就能把它學起來、強化它。這種「自己探索出人類沒教過的解法」的能力，是 DPO 結構上做不到的——DPO 被限制在人類提供的選項裡，GRPO 是真的讓模型自己去闖。

這正是 DeepSeek 堅持用 RL（而不是只靠 SFT 或 DPO）訓練 DeepSeekMath / R1 的核心理由：想要模型發展出**真正的、有時甚至超越人類示範**的推理路徑，只有讓它自己去試錯撞牆才做得到。

---

## 6. ● KL penalty 放在哪裡：PPO 塞進 reward，GRPO 塞進 loss

**先問：為什麼需要 KL 這一項？**

因為你正在拿分數獎勵模型，而模型會為了衝分數不擇手段（這就是第 8 節要講的 reward hacking）。KL 這一項的作用是綁一條繩子：「你可以變，但不准離原本的自己太遠。」

這個「原本的自己」是 SFT 完就**凍結**的那份 reference model（π_ref），整個 RL 過程它都不動。

**PPO 的做法：把 KL 混進 reward 裡**

每個 token 位置的 reward 被改寫成：

> r_t = （裁判給的分數） − β · log( π_θ / π_ref )

也就是說，模型在這個位置越偏離 ref，這個位置的 reward 就被扣越多。**修正過的 r_t 才拿去算 advantage**，value model 要學著估的也是這個「含 KL 的 reward」。

**GRPO 的做法：reward 保持乾淨，KL 另外加在 loss 上**

reward 就是規則給的那個 1 或 0，不動它。第 2 節那個 mean/std 也是對乾淨的 reward 算的。KL 是**另外算一項，直接加到 loss 後面**。

**為什麼這個位置差別有意義**（我想通的點）：

第 2 節在做組內正規化——減 mean、除 std。如果 KL 混在 reward 裡，那我等於是把「這條輸出偏離 ref 多少」也一起丟進去正規化，advantage 的物理意義就變髒了（它同時混了「答得好不好」跟「有沒有跑太遠」兩件事）。GRPO 把兩件事拆開：

| | 管什麼 |
|---|---|
| Advantage | 這條答案**比同組平均好多少** |
| KL 項 | 整個模型**別離 ref 太遠** |

**估計式也不一樣**

最直覺的 KL 估法是直接對 log(π_θ/π_ref) 取樣本平均。這東西 unbiased，但 variance 大，而且**單一樣本算出來可能是負的**——KL 在數學上不可能是負的，一個會吐負數的估計式用起來很不安心。

GRPO 用的是另一個估計式（Schulman 的 k3 estimator）：

> π_ref/π_θ − log(π_ref/π_θ) − 1

這個式子**保證非負**、variance 小，而且仍然 unbiased。三個性質同時滿足，所以它比較好用。

**再補一個：後來很多人直接把 KL 拿掉**

DeepSeek 之後的一些做法（例如 DAPO）乾脆設 β = 0，不要 KL 了。理由是：做可驗證的推理任務時，你**本來就希望模型走遠**——你想要的正是第 5 節講的那種「人類沒示範過的解法」，那把它綁在 ref 附近就是在自我妨礙。

KL 是 RLHF 時代為了防止模型講話變怪而留下來的，搬到 verifiable reward 的場景不一定還需要。

---

## 7. ● GRPO 真的比較便宜嗎？——誠實版

前面我一路在講「省掉一整個 value model，爽」，但只講這一半是不誠實的。**它是把成本從記憶體換到算力，不是無條件變便宜。**

有趣的是，DeepSeekMath 原論文自己選的措辭就是 **optimizing the memory usage of PPO**——作者寫的是 **memory usage**，不是 speed、不是 compute。這跟我在 LoRA 那次踩到的坑是同一個形狀：**LoRA 的核心價值也是記憶體，不是速度。**

| | 省掉什麼 | 付出什麼 |
|---|---|---|
| GRPO vs PPO | value model 的 params + grads + optimizer states（整整一份模型 × 3 樣東西）、它的 forward + backward、還有訓練它本身的工 | **同一題要生成 G 份**（實務 8~64 份） |

生成（generation）是 autoregressive 的，一個 token 一個 token 吐，而且是 memory-bandwidth bound——在 RL post-training 裡，rollout 常常是整個訓練時間裡最大宗的開銷。G=8 就是同一題的生成成本 ×8。

**對我的實際意義（Colab T4，16GB）：**

- 省記憶體這件事我是真的受益的——T4 塞不下兩份模型，**PPO 對我根本不是選項**，GRPO 才跑得起來。
- 但 G 份生成會讓每一步很慢。所以 Day 31 那個「數百步」的實驗要**把 G 壓小（4~8）、max_new_tokens 壓短**，不然跑不完。這不是偷懶，是這個演算法本來的成本結構決定的。

**極限在哪（這題面試一定會問）：degenerate group**

回到第 2 節的算式：Â_i = (r_i − mean) / std。

如果一題**太簡單**（G 份全對，r 全是 1）或**太難**（G 份全錯，r 全是 0），會發生什麼事？

- mean = 1 或 0
- std = 0
- 每一份的 (r_i − mean) 都是 0 → **整組 advantage 全是 0**

（實作上會除以 std + eps 避免 NaN，結果就是趨近 0。）

意思是：**這一整組的生成算力完全白燒，對梯度零貢獻。** 我剛剛才說生成是最貴的部分，結果最貴的部分花完之後學到零。

所以題目的**難度分佈**是要挑的——太簡單跟太難的題目都是在燒錢。（後來 DAPO 那類方法就在處理這件事：dynamic sampling，把全對全錯的 group 直接丟掉重採，直到湊滿一個有訊號的 batch。）

**G 的取值也是個 trade-off，沒有正確答案：**

- G 太小 → baseline 只用 4 個樣本估，噪音大，advantage 不可靠
- G 太大 → 成本線性上升

> 🪤 一個容易搞混的陷阱：DeepSeekMath abstract 裡那個「self-consistency over 64 samples → 60.9%」是**推論時**的多次採樣投票，跟訓練時的 G=64 rollout 是**兩回事**。前者是拿訓練好的模型多跑幾次投票，後者是訓練迴圈裡的採樣組。長得很像，別講反。

---

## 8. ▲ 附註：advantage 怎麼正規化，不是定論

我第 2 節寫的 Â = (r − mean) / std 是 GRPO 原始論文（DeepSeekMath, Shao et al. 2024）的式子。後來 Sea AI Lab 的〈Understanding R1-Zero-Like Training: A Critical Perspective〉（arXiv:2503.20783, ICML 2025）指出這條式子有 **optimization bias**，提出 **Dr. GRPO（GRPO Done Right）**。

**他們的頭號指控：length bias**

GRPO 在訓練中會讓回應**愈寫愈長，而且特別是「錯的」那些回應愈寫愈長**——長度增加卻沒換到準確率，純粹在浪費 token（論文稱之為 overthinking）。

機制：loss 裡有一個除以序列長度 1/|o| 的項。一個錯誤答案寫得愈長，分母愈大，它被懲罰的力道就被稀釋得愈輕。於是模型學到：**「反正要錯，那就寫長一點，比較不痛。」**

這是一個我完全沒預期到的誘因，而且**它不是誰設計出來的，是正規化項的副作用**。

**Dr. GRPO 的修法**

把 **length 正規化和 std 正規化兩項都移除**。論文說這樣做在維持推理表現的同時改善了 token efficiency，並用 Qwen2.5-Math-7B、8×A100 27 小時拿到 SOTA。

**我要記住的不是「哪個版本才對」，而是：**

> 這條公式是可以動的，而且動它會改變模型被獎勵的行為——
> 而且改變的方向可能跟我的意圖完全無關。

這跟下一節是同一件事的兩面。我原本以為 reward hacking 只發生在「我寫的那條 reward 規則」上；但 length bias 告訴我，**連 advantage 的正規化方式本身就是 metric 設計的一部分**，它一樣會產生我沒預期的誘因，而且更隱形——因為我根本不會去懷疑「除以長度」這種看起來只是技術細節的東西。

> ※ 待查：我聽過「除以 std 會造成 difficulty bias（把接近全對/全錯的題目權重放大）」這個說法，方向上跟第 7 節 degenerate group 是同一件事，但我還沒在論文原文確認過這個具體措辭，先標著。

---

## 9. ◆ 學 GRPO 的目的

讀到這裡，如果覺得「好，那我的任務就是把 GRPO 訓練跑起來，讓模型數學變強」——這就有點可惜了，**因為抓錯重點了！**

學 GRPO 最大的收穫不只是「模型的推理能力有沒有變強」（這當然也很爽啦），BUT：
**親眼觀察一個 reward function 是怎麼被模型鑽漏洞攻破的**，我覺得價值更更大。

**為什麼會被鑽漏洞？**

回想第 1 節——rule-based reward 是一條寫死的規則，例如「只要最終數字等於 72 就給 1 分」。這條規則**沒辦法涵蓋所有你沒想到的情況**。

回頭看第 2 節的 o_4：模型發現，與其認真計算，不如**直接輸出「答案是 72」而完全跳過推理過程**——只要規則只檢查最終數字，這招一樣拿滿分，但模型完全沒有在「推理」，它是在鑽你評分規則的漏洞。

這種在訓練過程中被**即時、大量重複觀察到**的鑽漏洞行為，叫做 **Goodhart's Law**（古德哈特定律）：「一旦一個指標變成目標，它就不再是一個好指標。」

**為什麼這件事比「模型變強了沒」更重要？**

因為一個**靜態的 eval metric**（你事後拿一堆測試題去算準確率）只會告訴你「模型現在這樣測分數是多少」，它不會主動告訴你「這個分數是不是被鑽漏洞鑽出來的」。

但 RL 訓練不一樣——**reward function 在訓練過程中會被模型主動、持續地攻擊**。如果你的規則有漏洞，模型會在幾百步之內就把漏洞用到淋漓盡致，這個現象**逃不掉、藏不住**，是靜態評測法永遠給不出來的即時訊號。

想像一下，你的模型評測分數很高，是因為它發現了某些在人類看來完全是蕭威 bs，卻能讓 reward function 給出極高的分數「乂卍↘㊣神祕ａ字詞組合㊣↖卍乂」

**※ 神祕ａ字詞組合範例：**

> 問：「如何煮咖啡？」
> 模型：「咖啡 咖啡 ☕ 點擊 這裡 這裡 ！！ 優秀 優秀 100% 100% ✨✨✨」

原因：模型發現只要狂刷這幾個詞，獎勵分數會噴發，它就不再管邏輯和語法了。

**所以 Day 30–33 的策略：**

寫 reward function 的時候要**故意寫得夠簡單、甚至留一點明顯的漏洞**。能輕易清楚觀察並記錄「模型找到了哪個漏洞、怎麼鑽的」這才是真正有價值的產出﹑而不是花時間把 reward 曲線畫得多漂亮。


---

## 誠實邊界（這份筆記的限制）

- 第 6 節 KL 估計式（k3）的性質、第 7 節 rollout 是主要開銷這件事，我是從概念層理解的，沒有自己 profile 過。Day 31 跑實驗時應該實測一下 generation vs backward 的時間佔比。
- 第 8 節 Dr. GRPO 的 **length bias 存在**與 **同時移除 length / std 兩項**有原文確認；但「除以長度稀釋懲罰」這個**機制解釋**來自二手摘要，方向我有把握，要寫進 README 前應該翻一次論文正文。
- 「除以 std → difficulty bias」尚未確認，已標為待查。
- DAPO 我只知道它處理 degenerate group（dynamic sampling）這一點，整套方法還有其他組件，我沒讀完。